# Traditional Methods - Code

---

## Modeling Approach

Our baseline CNN uses a simple three-layer architecture that serves as a foundational benchmark for diabetic retinopathy classification. The model progressively extracts features through three convolutional blocks (32, 64, and 128 channels) with 3x3 kernels, ReLU activation, and max pooling, reducing input images from 224x224 to 28x28 while learning increasingly complex patterns from basic edges to pathological features. The extracted features are then flattened and passed through two fully connected layers (256 neurons, then 5 output classes) with dropout regularization to prevent overfitting. This straightforward design allows us to establish baseline performance and evaluate whether the added complexity of modern architectures like ResNet and EfficientNet provides meaningful improvements for retinal image classification, while maintaining transparency and interpretability.

## Model Components

````{tab-set}
```{tab-item} Convolutional Layer
- Applies filters across the input image to detect local patterns like edges, textures, or shapes, forming the building blocks of visual understanding.
```
```{tab-item} Max Pooling
- Reduces the spatial dimensions by taking the maximum value within small regions, helping to retain important features while making the model more efficient and robust.
```

```{tab-item} ReLU Activation
- $\text{ReLU}(x) = \max(0, x)$
- Applies a non-linear function that replaces negative values with zero, allowing the network to learn complex patterns while keeping computations simple.
```
````

After downloading the images and labels from the drive, we will define and train our CNN using PyTorch.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, TensorDataset
import copy
import random

In [2]:
# Set seeds for reproducibility

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

```{admonition} Note
The code blocks below were run on a GPU, which significantly speeds up model training. While using a GPU is recommended, the code should still run on a CPU as well.
```

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
x_train = torch.load('data/x_train.pt')
y_train = torch.load('data/y_train.pt')
x_test = torch.load('data/x_test.pt')
y_test = torch.load('data/y_test.pt')

In [ ]:
# Define the CNN model

class CNN(nn.Module):
    def __init__(self, num_classes=5):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

### Cost-Sensitive Learning

Due to the class imbalance in our dataset, we need to ensure that our models don't bias predictions toward the majority class (0 - no DR), which would hurt performance on rare but clinically important cases. We will use cost-sensitive learning, which addresses this problem by assigning higher penalties to misclassifying underrepresented classes. The function below creates class weights, which tell our criterion how to adjust loss contribution based on class representation.

In [6]:
def get_class_weights(y_train, device):
    label_counts = Counter(y_train.tolist())
    num_classes = len(label_counts)
    total_samples = sum(label_counts.values())
    
    class_weights = [total_samples / (num_classes * label_counts[i]) for i in range(num_classes)]
    return torch.tensor(class_weights, dtype=torch.float).to(device)

## Model Training

We will use [K-fold cross validation](https://aiml.com/what-are-the-subtypes-of-cross-validation/), which tests performance on various sections of our training set, as we train and tune the hyperparameters of our model. Once we find hyperparameters that seem optimal, we will evaluate the model on our test set. We encourage you to tweak the hyperparameters below to see how they affect model performance.

In [ ]:
# Model Hyperparameters (Cross Validation)

cv_epochs = 20
cv_patience = 5
cv_batch_size = 32
cv_lr = 0.001

The following function trains our CNN using a training and validation set, with support for early stopping. It tracks the best validation loss and associated metrics (QWK, precision, recall, F1) across epochs. The model is updated using the provided loss function and optimizer, and early stopping is triggered if validation loss doesn't improve for a specified number of epochs. After training, it returns the best-performing model and its validation metrics.

In [8]:
def train_model(model_class, train_loader, val_loader, criterion, optimizer, device, epochs=cv_epochs, patience=cv_patience):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    model = model_class.to(device)
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                
                # Collect predictions and labels for metrics
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
        # Early stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            
            # Calculate validation metrics for the best epoch
            val_labels_np = np.array(val_labels)
            val_preds_np = np.array(val_preds)
            
            qwk = cohen_kappa_score(val_labels_np, val_preds_np, weights='quadratic')
            precision, recall, f1, _ = precision_recall_fscore_support(
                val_labels_np, val_preds_np, average='weighted', zero_division=0
            )
            
            best_val_metrics = {
                'qwk': qwk,
                'precision': precision,
                'recall': recall,
                'f1': f1
            }
            
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs)")
                break
    
    # Load best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation loss: {best_val_loss:.4f}")
    
    return model, best_val_metrics

:::{dropdown} Code Explanation
```python
def train_model(model_class, train_loader, val_loader, criterion, optimizer, device, epochs=cv_epochs, patience=cv_patience):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    model = model_class.to(device)
```
Initializes training by setting up early stopping variables, tracking the best validation loss and metrics, and moving the model to the appropriate device.

```python
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
```
Training phase: for each epoch, the model is trained on all batches in the training loader. Loss is accumulated to track average training performance.

```python
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
```
Validation phase: the model is evaluated on the validation set without updating weights. It computes validation loss and collects predictions and true labels for metric computation.

```python
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())

            val_labels_np = np.array(val_labels)
            val_preds_np = np.array(val_preds)

            qwk = cohen_kappa_score(val_labels_np, val_preds_np, weights='quadratic')
            precision, recall, f1, _ = precision_recall_fscore_support(
                val_labels_np, val_preds_np, average='weighted', zero_division=0
            )

            best_val_metrics = {
                'qwk': qwk,
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

            epochs_no_improve = 0
```
If the current validation loss is the best so far, save the model weights and compute evaluation metrics: quadratic weighted kappa, precision, recall, and F1-score.

```python
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs)")
                break
```
Early stopping: if the validation loss hasn't improved for `patience` consecutive epochs, stop training early.

```python
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation loss: {best_val_loss:.4f}")

    return model, best_val_metrics
```
After training, the best model weights are reloaded and both the model and the best validation metrics are returned.
:::


The following function performs cross-validation to evaluate our CNN. It uses StratifiedKFold to preserve class distributions across folds and trains a new model on each split using cost-sensitive learning and early stopping. After each fold, it records performance metrics (QWK, precision, recall, F1-score) on the validation set. Finally, it prints and returns the average metrics across all folds to summarize model performance.

In [9]:
def cross_validate(model_class, X, y, n_splits=5, epochs=cv_epochs, patience=cv_patience, batch_size=cv_batch_size, lr=cv_lr):
    print(f"Using device: {device}")
    print(f"Dataset: {len(X)} images")
    
    # Convert to numpy
    X = X.numpy()
    y = y.numpy()
    
    # Initialize cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Store results
    cv_results = {
        'qwk_scores': [],
        'precision_scores': [],
        'recall_scores': [],
        'f1_scores': []
    }
    
    # Cross-validation loop
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")
        
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Convert back to tensors
        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        y_train = torch.LongTensor(y_train)
        y_val = torch.LongTensor(y_val)
        
        # Create data loaders
        train_loader = DataLoader(TensorDataset(X_train, y_train), 
                                batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val, y_val), 
                               batch_size=batch_size, shuffle=False)

        # Initialize model
        model = model_class.to(device)
        
        # Calculate class weights for cost-sensitive learning
        class_weights = get_class_weights(y_train, device)
        print(f"Class weights: {class_weights.cpu().numpy()}")
        
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        # Train model with early stopping
        model, val_metrics = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, patience)
        
        # Use validation metrics from training
        qwk = val_metrics['qwk']
        precision = val_metrics['precision']
        recall = val_metrics['recall']
        f1 = val_metrics['f1']
        
        print(f"Fold {fold+1} Results:")
        print(f"  QWK: {qwk:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-score: {f1:.4f}")
        
        # Store results
        cv_results['qwk_scores'].append(qwk)
        cv_results['precision_scores'].append(precision)
        cv_results['recall_scores'].append(recall)
        cv_results['f1_scores'].append(f1)
    
    # Calculate and print summary statistics
    print(f"\n{'='*50}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*50}")
    
    for metric, scores in cv_results.items():
        metric_name = metric.replace('_scores', '').replace('_', ' ').title()
        mean_score = np.mean(scores)
        print(f"{metric_name:12}: {mean_score:.4f}")
    
    return cv_results

:::{dropdown} Code Explanation
```python
def cross_validate(model_class, X, y, n_splits=5, epochs=cv_epochs, patience=cv_patience, batch_size=cv_batch_size, lr=cv_lr):
    print(f"Using device: {device}")
    print(f"Dataset: {len(X)} images")
    
    X = X.numpy()
    y = y.numpy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_results = {
        'qwk_scores': [],
        'precision_scores': [],
        'recall_scores': [],
        'f1_scores': []
    }
```
Prepares for cross-validation by converting input tensors to NumPy arrays and initializing a `StratifiedKFold` splitter. An empty dictionary is created to collect metrics across folds.

```python
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        y_train = torch.LongTensor(y_train)
        y_val = torch.LongTensor(y_val)

        train_loader = DataLoader(TensorDataset(X_train, y_train), 
                                  batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val, y_val), 
                                batch_size=batch_size, shuffle=False)
```
Splits the data into training and validation sets for the current fold, converts them back into PyTorch tensors, and wraps them in DataLoaders for mini-batch training and evaluation.

```python
        model = model_class.to(device)

        class_weights = get_class_weights(y_train, device)
        print(f"Class weights: {class_weights.cpu().numpy()}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        model, val_metrics = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, patience)
```
Initializes a fresh instance of the model, computes class weights to handle class imbalance, sets up the loss and optimizer, and trains the model using early stopping. Validation metrics are returned from `train_model`.

```python
        qwk = val_metrics['qwk']
        precision = val_metrics['precision']
        recall = val_metrics['recall']
        f1 = val_metrics['f1']

        print(f"Fold {fold+1} Results:")
        print(f"  QWK: {qwk:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-score: {f1:.4f}")

        cv_results['qwk_scores'].append(qwk)
        cv_results['precision_scores'].append(precision)
        cv_results['recall_scores'].append(recall)
        cv_results['f1_scores'].append(f1)
```
Extracts the validation metrics from the training result and logs them for this fold. The metrics are also saved to the `cv_results` dictionary for later summarization.

```python
    print(f"\n{'='*50}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*50}")

    for metric, scores in cv_results.items():
        metric_name = metric.replace('_scores', '').replace('_', ' ').title()
        mean_score = np.mean(scores)
        print(f"{metric_name:12}: {mean_score:.4f}")

    return cv_results
```
After all folds are complete, computes and prints average scores for each metric across all folds. The results are returned for downstream use.
:::


In [10]:
model_class = CNN()
cv_results = cross_validate(model_class, x_train, y_train)

Using device: cuda
Dataset: 2929 images

--- Fold 1/5 ---
Class weights: [0.40571427 1.9772152  0.73333335 3.809756   2.479365  ]
Epoch 1/20, Train Loss: 1.4665, Val Loss: 1.4258
Epoch 2/20, Train Loss: 1.2951, Val Loss: 1.2663
Epoch 3/20, Train Loss: 1.2314, Val Loss: 1.2867
Epoch 4/20, Train Loss: 1.1830, Val Loss: 1.2431
Epoch 5/20, Train Loss: 1.1415, Val Loss: 1.2256
Epoch 6/20, Train Loss: 1.1256, Val Loss: 1.2230
Epoch 7/20, Train Loss: 1.0909, Val Loss: 1.2167
Epoch 8/20, Train Loss: 1.0296, Val Loss: 1.1607
Epoch 9/20, Train Loss: 0.9618, Val Loss: 1.2215
Epoch 10/20, Train Loss: 0.8878, Val Loss: 1.2298
Epoch 11/20, Train Loss: 0.7911, Val Loss: 1.2194
Epoch 12/20, Train Loss: 0.7283, Val Loss: 1.2189
Epoch 13/20, Train Loss: 0.6681, Val Loss: 1.2588
Early stopping triggered after 13 epochs (no improvement for 5 epochs)
Loaded best model with validation loss: 1.1607
Fold 1 Results:
  QWK: 0.6150
  Precision: 0.7073
  Recall: 0.6638
  F1-score: 0.6753

--- Fold 2/5 ---
Class w

Solid initial results! While some folds were much more difficult to classify than others, our average performance shows decent scores, with a quadratic weighted kappa of 0.7656, precision of 0.8091, recall of 0.7737, and F1-score of 0.7803. These are well-balanced and indicate that our model was able to identify features in the retina images that correspond to diabetic retinopathy. 

## Evaluation on Test Set

After running several cross validation experiments with different hyperparameters, we will evaluate our CNN on the test dataset to record our final evaluation. Based on early stopping trends during cross validation, we will train for 8 epochs. Other hyperparameters are listed below.

In [ ]:
# Model Hyperparameters (Final Evaluation)

epochs = 8
batch_size = 32
lr = 0.001

In [12]:
def evaluate_final_model(model_class, X_train, y_train, X_test, y_test, epochs=epochs, batch_size=batch_size, lr=lr):        
    train_loader = DataLoader(TensorDataset(X_train, y_train), 
                            batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), 
                           batch_size=batch_size, shuffle=False)

    # Initialize model
    model = model_class.to(device)
    
    # Setup training
    class_weights = get_class_weights(y_train, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Train on entire train set (no early stopping)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluate on test set
    print(f"\nEvaluating on test set ({len(X_test)} samples)...")
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    
    # Calculate metrics
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    print(f"\nFinal Test Results:")
    print(f"  QWK: {qwk:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    return model, y_true, y_pred

:::{dropdown} Code Explanation
```python
def evaluate_final_model(model_class, X_train, y_train, X_test, y_test, epochs=epochs, batch_size=batch_size, lr=lr):        
    train_loader = DataLoader(TensorDataset(X_train, y_train), 
                            batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), 
                           batch_size=batch_size, shuffle=False)
```
Wraps the full training and test datasets in PyTorch `DataLoader` objects for batched training and evaluation. Training data is shuffled; test data is not.

```python
    model = model_class.to(device)
    
    class_weights = get_class_weights(y_train, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
```
Initializes the model and training components: class-weighted loss function (for imbalanced classes), and the Adam optimizer with the specified learning rate.

```python
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
```
Trains the model for the specified number of epochs on the full training set. Unlike earlier training loops, this does not include early stopping or validation — it's a final fit.

```python
    print(f"\nEvaluating on test set ({len(X_test)} samples)...")
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
```
Evaluates the trained model on the test set. Predictions and true labels are collected across all batches and converted into NumPy arrays for metric calculation.

```python
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    print(f"\nFinal Test Results:")
    print(f"  QWK: {qwk:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    return model, y_true, y_pred
```
Computes final evaluation metrics: Quadratic Weighted Kappa, Precision, Recall, and F1-score. These are printed and returned along with the model and test predictions.
:::


In [13]:
model, y_true, y_pred = evaluate_final_model(model_class, x_train, y_train, x_test, y_test)

Epoch 1/8, Loss: 0.4833
Epoch 2/8, Loss: 0.3941
Epoch 3/8, Loss: 0.3958
Epoch 4/8, Loss: 0.3336
Epoch 5/8, Loss: 0.2806
Epoch 6/8, Loss: 0.3009
Epoch 7/8, Loss: 0.2720
Epoch 8/8, Loss: 0.2290

Evaluating on test set (733 samples)...

Final Test Results:
  QWK: 0.7074, Precision: 0.7217, Recall: 0.7135, F1: 0.7120


Our final test results are a bit weaker than our cross validation results, which is reasonable. Since we tweaked the hyperparameters according to our cross validation results, they're optimized for the training set and not our test set. Early stopping also contributes to better cross validation performance as we select the model with the best validation results. Overall, these results still indicate our model is identifying useful features and generally assigning them to the correct classes. You're encouraged to change model parameters and the CNN architecture to improve performance. In our results section, we'll dive deeper into our results and analyze performance class-by-class.

In [ ]:
# Save the final model state for results section

torch.save(model.state_dict(), 'cnn_state_dict.pth')